In [6]:
import os
import pandas as pd
import numpy as np

def analyze_customer_leak_with_type(file_path, folder_type, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Processes any customer dataset. Threshold criteria adapt directly
    to the folder name provided: 'Villa (Residential)', 'Commercial', etc.
    """
    try:
        # --- 1. Load, Sort, and Clean Data ---
        df = pd.read_excel(file_path, header=2)
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
        
        # Extract Time Features
        df['hour'] = df[time_col].dt.hour
        df['day_of_week'] = df[time_col].dt.dayofweek
        
        # Timeline Splitting
        max_date = df[time_col].max()
        min_date = df[time_col].min()
        four_weeks_ago = max_date - pd.Timedelta(weeks=4)
        first_month_end = min_date + pd.Timedelta(days=30)
        
        df['period'] = np.where(df[time_col] >= four_weeks_ago, 'Recent', 'Middle')
        df.loc[df[time_col] <= first_month_end, 'period'] = 'Historical'
        
        # --- 2. Build Baseline Profiles ---
        recent_data = df[df['period'] == 'Recent']
        recent_profile = recent_data.groupby(['day_of_week', 'hour'])[consumption_col].median().reset_index()
        recent_profile.rename(columns={consumption_col: 'baseline_median'}, inplace=True)
        recent_std = recent_data.groupby(['day_of_week', 'hour'])[consumption_col].std().reset_index()
        recent_std.rename(columns={consumption_col: 'baseline_std'}, inplace=True)
        
        df = pd.merge(df, recent_profile, on=['day_of_week', 'hour'], how='left')
        df = pd.merge(df, recent_std, on=['day_of_week', 'hour'], how='left')
        
        # --- 3. Compute Structural Deviations ---
        df['abs_deviation'] = df[consumption_col] - df['baseline_median']
        df['z_score'] = df['abs_deviation'] / (df['baseline_std'] + 1e-5)
        df[['abs_deviation', 'z_score']] = df[['abs_deviation', 'z_score']].fillna(0)
        
        # --- 4. DYNAMIC FOLDER-BASED THRESHOLD ASSIGNMENT ---
        # Fixed explicit arrays to ensure Python executes them safely
        if any(kw in folder_type for kw in ["Residential", "Villa", "Flat"]):
            night_hours = [1, 2, 3, 4]
            night_drift_threshold = 0.025     # Lower barrier for residential (25 liters)
            z_threshold = 3.0                 
            consecutive_hours = 2             
            
        elif "Government" in folder_type:
            night_hours = [0, 1, 5, 6]        # Safely jumps 2-4 AM irrigation
            night_drift_threshold = 0.500     # 500 liters minimum shift
            z_threshold = 4.5                 
            consecutive_hours = 5             
            
        elif "Industrial" in folder_type:
            night_hours = [2, 3, 4]              
            night_drift_threshold = 1.000     # 1000 liters floor
            z_threshold = 5.0                 
            consecutive_hours = 6             
            
        else: # "Commercial" or "Hotel"
            night_hours = [1, 2, 3, 4]
            night_drift_threshold = 0.150     # 150 liters change allowance
            z_threshold = 3.5
            consecutive_hours = 4

        # --- 5. Evaluate Automation Logic ---
        leak_detected = False
        leak_reasons = []
        
        hist_night = df[(df['period'] == 'Historical') & (df['hour'].isin(night_hours))]
        recent_week_night = df[(df[time_col] >= (max_date - pd.Timedelta(days=7))) & (df['hour'].isin(night_hours))]
        
        historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0
        recent_min_flow = recent_week_night[consumption_col].min() if not recent_week_night.empty else 0
        
        # FIXED UNIFIED NIGHT FLOW LOGIC: 
        # Checks if current floor is higher than history AND has moved past our safety threshold
        if recent_min_flow > (historical_min_flow + night_drift_threshold):
            # Additional sanity check: rules out tiny variance if background is strictly zero
            if recent_min_flow > 0.03: 
                leak_detected = True
                leak_reasons.append(f"Slow Leak: Constant minimum night baseline has lifted from {historical_min_flow:.3f} m3 up to {recent_min_flow:.3f} m3.")
        
        # Daytime Burst Evaluation
        df['is_anomaly'] = (df['period'] == 'Recent') & (df['z_score'] > z_threshold)
        consecutive_anomalies = df['is_anomaly'].astype(int).rolling(window=consecutive_hours).sum()
        
        if (consecutive_anomalies >= consecutive_hours).any():
            leak_detected = True
            max_z = df.loc[df['period'] == 'Recent', 'z_score'].max()
            leak_reasons.append(f"Sudden Burst: High usage spike sustained for {consecutive_hours}+ hours (Peak Z-Score: {max_z:.1f}).")
            
        # --- 6. Output Summary Dashboard ---
        print("="*85)
        print(f"📋 AUTOMATED LEAK AUDIT REPORT: {os.path.basename(file_path)}")
        print(f"Subfolder Profile Read: {folder_type.upper()}")
        print("="*85)
        
        if leak_detected:
            print("🔴 STATUS: LEAK SUSPECTED")
            for reason in leak_reasons:
                print(f"  - {reason}")
        else:
            print("🟢 STATUS: NORMAL (NO LEAK DETECTED)")
        print("="*85 + "\n")
        
        return {
            "Filename": os.path.basename(file_path),
            "Profile_Folder": folder_type,
            "Leak_Suspected": "YES" if leak_detected else "NO",
            "Details": "; ".join(leak_reasons) if leak_reasons else "Normal usage patterns.",
            "Historical_Night_Min": round(historical_min_flow, 4),
            "Recent_Night_Min": round(recent_min_flow, 4)
        }
        
    except Exception as e:
        print(f"❌ Failure for file {file_path}: {str(e)}\n")
        return {
            "Filename": os.path.basename(file_path),
            "Profile_Folder": folder_type,
            "Leak_Suspected": "ERROR",
            "Details": f"Processing failure: {str(e)}",
            "Historical_Night_Min": 0,
            "Recent_Night_Min": 0
        }


In [ ]:
# 1. Run the cleaner pipeline on your 'datasets' folder tree
all_audit_results = run_portfolio_leak_audit(base_folder="customers")

# 2. Automatically transform results array into a structured DataFrame
summary_df = pd.DataFrame(all_audit_results)

# 3. Export directly to a master tracking CSV sheet in your project workspace
output_file = "portfolio_leakage_audit_summary.csv"
summary_df.to_csv(output_file, index=False)

print(f"💾 SUCCESS: Master audit log ledger exported to: '{output_file}'!")
summary_df.head()
